# Setup and Environment Configuration

In [ ]:
from truveta.study import Client, OutputMode, display_df
import pyspark.pandas as ps
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pyspark.sql.functions import col, floor, datediff, to_date

client = Client(output_mode = OutputMode.PandasOnSpark)
study = client.get_study()

In [ ]:
population = study.get_population(title = "Aim1b_Female_Update")
snapshot = population.get_latest_snapshot()

In [ ]:
snapshot.get_tables()

# Cohort Index Date Loading

In [ ]:
output_path_spark = study.get_output_path()

# T2D Primary Population
t2d_primary = output_path_spark + "/output_data/T2D_Primary_basedata_Mar10"
loaded_index_date = spark.read.parquet(t2d_primary)
loaded_index_date = loaded_index_date.filter(loaded_index_date.Sex == 1065405)
loaded_index_date = loaded_index_date.withColumnRenamed("FirstMedTime", "index_date")
loaded_index_date = loaded_index_date.dropDuplicates()
loaded_index_date.createOrReplaceTempView('tbl_loaded_index_date')

# T2D Secondary Population
t2d_secondary = output_path_spark + "/output_data/T2D_Secondary_basedata_Mar10"
loaded_index_date = spark.read.parquet(t2d_secondary)
loaded_index_date = loaded_index_date.filter(loaded_index_date.Sex == 1065405)
loaded_index_date = loaded_index_date.withColumnRenamed("FirstMedTime", "index_date")
loaded_index_date = loaded_index_date.dropDuplicates()
loaded_index_date.createOrReplaceTempView('tbl_loaded_index_date')

In [ ]:
# Create Lookback Period Relative to Index Date
lookback_query = """
SELECT PersonId, index_date, CAST(DATEADD(YEAR, -2, index_date) AS DATE) AS look_back
FROM tbl_loaded_index_date
"""
patient_index_date = spark.sql(lookback_query)
patient_index_date.createOrReplaceTempView("tbl_patient_index_date")

# Mortality Ascertainment


## Death Record Extraction

In [ ]:
# Check Patient’s Death Datetime
death_query = """
SELECT
    ind.PersonId,
    MIN(CAST(d.DeathDateTime AS Date)) AS Death_Time
FROM tbl_patient_index_date ind
LEFT JOIN PersonDeathFact d ON d.PersonId = ind.PersonId
GROUP BY ind.PersonId
"""
death = spark.sql(death_query)
death.createOrReplaceTempView("tbl_death")

## Death Date Adjudication Algorithm

In [ ]:
# Last 3 Encounters
last_three_encounters_query = """
WITH RankedEncounters AS (
SELECT
  PersonId,
  CAST(StartDateTime AS DATE) AS EncounterDate,
  DENSE_RANK() OVER (PARTITION BY PersonId ORDER BY StartDateTime DESC) AS EncounterRank
FROM Encounter e
  WHERE
   StatusConceptId NOT IN (2506590, 2506591, 2506595, 2506597, 2983200, 2983199)
   AND TypeConceptId NOT IN (3059298, 3059269, 3059303, 3059266, 3059301, 3059302, 3059273, 3059282, 3059297, 3059261, 3059274, 1065339, 3059290, 3059262)
   AND ClassConceptId NOT IN (1065226)
   AND PersonId IN (SELECT PersonId FROM tbl_patient_index_date)
   AND StartDateTime <= '2025-10-06'
)
SELECT
  PersonId AS LE_PersonId,
  MAX(CASE WHEN EncounterRank=1 THEN EncounterDate END) AS LastEncounterDate,
  MAX(CASE WHEN EncounterRank=2 THEN EncounterDate END) AS Sec2LastEncounterDate,
  MAX(CASE WHEN EncounterRank=3 THEN EncounterDate END) AS Third2LastEncounterDate
FROM
  RankedEncounters
WHERE
  EncounterRank <= 3
GROUP BY
  PersonId
ORDER BY
  PersonId
"""
last_three_encounters = spark.sql(last_three_encounters_query)
last_three_encounters.createOrReplaceTempView('tbl_last_three_encounters')

In [ ]:
# Clean mortality data
cleaned_mortality_query = """
SELECT
   d.PersonId,
   d.DeathDateTime,
   d.ConfidenceScore,
   c1.ConceptName AS DeceasedDataSource,
   c2.ConceptName AS DeathVerifiedProofCode,
   c3.ConceptName AS SourceConcept,
   d.DeceasedDataSourceConceptId,
   d.DeathVerifiedProofCodeConceptId,
   d.SourceConceptId
FROM
   PersonDeathFact d
LEFT JOIN
   Concept c1 ON d.DeceasedDataSourceConceptId = c1.ConceptId
LEFT JOIN
   Concept c2 ON d.DeathVerifiedProofCodeConceptId = c2.ConceptId
LEFT JOIN
   Concept c3 ON d.SourceConceptId = c3.ConceptId
WHERE DeathDateTime IS NOT NULL
"""
cleaned_mortality = spark.sql(cleaned_mortality_query)
cleaned_mortality.createOrReplaceTempView('tbl_cleaned_mortality')

In [ ]:
min_max_mortality_query = """
SELECT
    PersonId,
    MIN(DeathDateTime) AS MinDeathTime,
    MAX(DeathDateTime) AS MaxDeathTime,
    DATEDIFF(MAX(DeathDateTime), MIN(DeathDateTime)) AS DeathDateDiff
FROM tbl_cleaned_mortality
GROUP BY PersonId
"""

mortality_summary = spark.sql(min_max_mortality_query)
mortality_summary.createOrReplaceTempView('tbl_mortality_summary')

In [ ]:
persondeathfact_date_query = """
SELECT
    PersonId,
    CAST(DeathDateTime AS DATE) AS DeathDate,
    ConfidenceScore,
    DeceasedDataSourceConceptId,
    DeathVerifiedProofCodeConceptId,
    SourceConceptId
FROM tbl_cleaned_mortality
"""
spark.sql(persondeathfact_date_query).createOrReplaceTempView("PersonDeath")

deathdates_query = """
WITH DeathDates AS (
  SELECT DISTINCT
    PersonId, DeathDate,
    MAX(ConfidenceScore) AS ConfidenceScore,
    MAX(DeceasedDataSourceConceptId) AS DeceasedDataSourceConceptId,
    MAX(DeathVerifiedProofCodeConceptId) AS DeathVerifiedProofCodeConceptId,
    MAX(SourceConceptId) AS SourceConceptId
  FROM PersonDeath
  GROUP BY PersonId, DeathDate
),

DeathCounts AS (
  SELECT
    d.PersonId AS DeathPersonId,
    d.DeathDate,
    COUNT(*) OVER (PARTITION BY d.PersonId, d.DeathDate) AS Frequency,
    d.ConfidenceScore,
    c1.ConceptName AS DeceasedDataSource,
    c2.ConceptName AS DeathVerifiedProofCode,
    c3.ConceptName AS SourceConcept
  FROM DeathDates d
  LEFT JOIN Concept c1 ON d.DeceasedDataSourceConceptId = c1.ConceptId
  LEFT JOIN Concept c2 ON d.DeathVerifiedProofCodeConceptId = c2.ConceptId
  LEFT JOIN Concept c3 ON d.SourceConceptId = c3.ConceptId
),

RankedDeaths AS (
  SELECT
    DeathPersonId,
    DeathDate,
    Frequency,
    ConfidenceScore,
    DeceasedDataSource,
    DeathVerifiedProofCode,
    SourceConcept,
    ROW_NUMBER() OVER (
      PARTITION BY DeathPersonId
      ORDER BY Frequency DESC, DeathDate ASC
    ) AS rn
  FROM DeathCounts
),

RecentRankDeaths AS (
  SELECT
    g.PersonId AS DeathPersonId,
    g.DeathDate,
    g.ConfidenceScore,
    c1.ConceptName AS DeceasedDataSource,
    c2.ConceptName AS DeathVerifiedProofCode,
    c3.ConceptName AS SourceConcept,
    ROW_NUMBER() OVER (
      PARTITION BY g.PersonId
      ORDER BY g.DeathDate DESC
    ) AS rn
  FROM DeathDates g
  LEFT JOIN Concept c1 ON g.DeceasedDataSourceConceptId = c1.ConceptId
  LEFT JOIN Concept c2 ON g.DeathVerifiedProofCodeConceptId = c2.ConceptId
  LEFT JOIN Concept c3 ON g.SourceConceptId = c3.ConceptId
),

RecentDeaths AS (
  SELECT
    DeathPersonId, DeathDate, ConfidenceScore,
    DeceasedDataSource, DeathVerifiedProofCode, SourceConcept
  FROM RecentRankDeaths
  WHERE rn = 1
)

SELECT
  rd.DeathPersonId,
  MAX(CASE WHEN rd.rn = 1 THEN rd.DeathDate END) AS DeathDate_mode,
  MAX(CASE WHEN rd.rn = 1 THEN rd.SourceConcept END) AS SourceConcept_mode,
  MAX(CASE WHEN rd.rn = 2 THEN rd.DeathDate END) AS DeathDate_mode2,
  MAX(CASE WHEN rd.rn = 2 THEN rd.SourceConcept END) AS SourceConcept_mode2,
  MAX(CASE WHEN rd.rn = 3 THEN rd.DeathDate END) AS DeathDate_mode3,
  MAX(CASE WHEN rd.rn = 3 THEN rd.SourceConcept END) AS SourceConcept_mode3,
  red.DeathDate AS DeathDate_recent,
  red.SourceConcept AS SourceConcept_recent
FROM RankedDeaths rd
LEFT JOIN RecentDeaths red
  ON rd.DeathPersonId = red.DeathPersonId
GROUP BY rd.DeathPersonId, red.DeathDate, red.SourceConcept
"""

mortality_final = spark.sql(deathdates_query)
mortality_final.createOrReplaceTempView("tbl_final_mortality")

In [ ]:
km_s0_query = """
SELECT
  i.*,
  l.LastEncounterDate,
  l.Sec2LastEncounterDate as LastEncounterDate2,
  l.Third2LastEncounterDate as LastEncounterDate3,
  m.DeathDate_mode,
  m.SourceConcept_mode,
  m.DeathDate_mode2,
  m.SourceConcept_mode2,
  m.DeathDate_mode3,
  m.SourceConcept_mode3,
  m.DeathDate_recent,
  m.SourceConcept_recent
FROM
  tbl_patient_index_date i
LEFT JOIN
  tbl_last_three_encounters l ON l.LE_PersonId = i.PersonId
LEFT JOIN
  tbl_final_mortality m ON m.DeathPersonId = i.PersonId
"""
spark.sql(km_s0_query).createOrReplaceTempView("tbl_km0")

km_s_query = """
SELECT *,
    TIMESTAMPDIFF(DAY, DeathDate_mode, LastEncounterDate) as DeathmLE1Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode2, LastEncounterDate) as Deathm2LE1Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode3, LastEncounterDate) as Deathm3LE1Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode, LastEncounterDate2) as DeathmLE2Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode2, LastEncounterDate2) as Deathm2LE2Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode3, LastEncounterDate2) as Deathm3LE2Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode, LastEncounterDate3) as DeathmLE3Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode2, LastEncounterDate3) as Deathm2LE3Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode3, LastEncounterDate3) as Deathm3LE3Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode, DeathDate_mode2) as DeathmDeathm2Diff,
    TIMESTAMPDIFF(DAY, DeathDate_mode2, DeathDate_mode3) as Deathm2Deathm3Diff,
    TIMESTAMPDIFF(DAY, LastEncounterDate2, LastEncounterDate) AS LE1LE2Diff,
    TIMESTAMPDIFF(DAY, LastEncounterDate3, LastEncounterDate2) AS LE2LE3Diff
FROM tbl_km0
"""
spark.sql(km_s_query).createOrReplaceTempView("tbl_km")

km_clean_s_query = """
SELECT *,
  CASE
    WHEN DeathDate_mode2 IS NULL OR (DeathDate_mode >= LastEncounterDate OR LastEncounterDate IS NULL) THEN DeathDate_mode
    ELSE
      CASE
        WHEN LastEncounterDate IS NOT NULL THEN
          CASE
            WHEN DeathDate_mode < LastEncounterDate AND DeathDate_mode2 < LastEncounterDate AND DeathDate_mode2 IS NOT NULL THEN
              CASE WHEN DeathmLE1Diff > Deathm2LE1Diff THEN DeathDate_mode2 ELSE DeathDate_mode END
            ELSE DeathDate_mode
          END

        WHEN ABS(DeathmLE1Diff) <= 30 AND LastEncounterDate IS NOT NULL THEN
          CASE
            WHEN ABS(Deathm2LE1Diff) <= ABS(DeathmLE1Diff) AND ABS(Deathm2LE1Diff) <= ABS(Deathm3LE1Diff) THEN DeathDate_mode2
            WHEN ABS(Deathm3LE1Diff) <= ABS(DeathmLE1Diff) AND ABS(Deathm3LE1Diff) <= ABS(Deathm2LE1Diff) AND DeathDate_mode3 IS NOT NULL THEN DeathDate_mode3
            ELSE DeathDate_mode
          END

        WHEN ABS(DeathmLE1Diff) > 30 AND ABS(Deathm2LE1Diff) <= 30 AND DeathDate_mode2 IS NOT NULL THEN
          CASE
            WHEN ABS(Deathm2LE1Diff) <= ABS(Deathm3LE1Diff) OR DeathDate_mode3 IS NULL THEN DeathDate_mode2
            WHEN ABS(Deathm3LE1Diff) <= ABS(Deathm2LE1Diff) AND DeathDate_mode3 IS NOT NULL THEN DeathDate_mode3
            ELSE DeathDate_mode2
          END

        WHEN DeathmLE1Diff > Deathm2LE1Diff
             AND Deathm2LE1Diff > Deathm3LE1Diff
             AND Deathm3LE1Diff <= 365
             AND LastEncounterDate IS NOT NULL
             AND DeathDate_mode3 IS NOT NULL
        THEN DeathDate_mode3

        WHEN (LE1LE2Diff <= 30 AND DeathmLE2Diff <= 30) OR (Deathm2LE2Diff <= 30 AND DeathmLE2Diff >= -30) THEN
          CASE
            WHEN ABS(Deathm2LE2Diff) <= ABS(DeathmLE2Diff) AND ABS(Deathm2LE2Diff) <= ABS(Deathm3LE2Diff) THEN DeathDate_mode2
            WHEN ABS(Deathm3LE2Diff) <= ABS(DeathmLE2Diff) AND ABS(Deathm3LE2Diff) <= ABS(Deathm2LE2Diff) THEN DeathDate_mode3
            ELSE DeathDate_mode
          END

        WHEN LE1LE2Diff > 30 AND DeathmLE2Diff <= 30 AND DeathmLE2Diff >= -30 THEN
          CASE
            WHEN ABS(Deathm2LE2Diff) <= ABS(DeathmLE2Diff) AND ABS(Deathm2LE2Diff) <= ABS(Deathm3LE2Diff) THEN DeathDate_mode2
            WHEN ABS(Deathm3LE2Diff) <= ABS(DeathmLE2Diff) AND ABS(Deathm3LE2Diff) <= ABS(Deathm2LE2Diff) THEN DeathDate_mode3
            ELSE DeathDate_mode
          END

        WHEN LE1LE2Diff > 30 AND ABS(DeathmLE2Diff) > 30 AND Deathm2LE2Diff <= 30 AND Deathm2LE2Diff >= -30 THEN
          CASE
            WHEN ABS(Deathm2LE2Diff) <= ABS(Deathm3LE2Diff) THEN DeathDate_mode2
            WHEN ABS(Deathm3LE2Diff) <= ABS(Deathm2LE2Diff) THEN DeathDate_mode3
            ELSE DeathDate_mode
          END

        ELSE NULL
      END
  END AS DeathDate
FROM tbl_km
"""
spark.sql(km_clean_s_query).createOrReplaceTempView("tbl_km_clean")

In [ ]:
final_death_query = """
SELECT
    PersonId,
    DeathDate AS Death_Time
FROM tbl_km_clean
"""
final_death = spark.sql(final_death_query)
final_death.createOrReplaceTempView('tbl_death')

In [ ]:
final_death_query = """
SELECT
    PersonId,
    index_date,
    DeathDate AS Event_Time,
    'Death' AS Event_Name
FROM tbl_km_clean
"""
final_death = spark.sql(final_death_query)
final_death.createOrReplaceTempView('tbl_final_death')

In [ ]:
last_encounter_query = """
SELECT
    PersonId,
    GREATEST(LastEncounterDate, LastEncounterDate2, LastEncounterDate3) AS Last_Encounter_Date
FROM tbl_km0
GROUP BY PersonId, LastEncounterDate, LastEncounterDate2, LastEncounterDate3
"""
last_encounter = spark.sql(last_encounter_query)
last_encounter.createOrReplaceTempView('tbl_last_encounter')

## All-cause mortality endpoint construction

In [ ]:
death_outcome_time_status_query = """
WITH Censor_Calc AS (
    SELECT
        ind.PersonId,
        ind.index_date,
        event.Event_Time,
        enc.Last_Encounter_Date,
        LEAST(
            DATE '2026-03-06',
            CASE
                WHEN enc.Last_Encounter_Date > ind.index_date
                THEN DATEADD(year, 1, enc.Last_Encounter_Date)
                ELSE DATE '2026-03-06'
            END,
            DATEADD(year, 3, ind.index_date)
        ) AS Censor_Date

    FROM tbl_patient_index_date ind
    LEFT JOIN tbl_final_death event
        ON ind.PersonId = event.PersonId
    LEFT JOIN tbl_last_encounter enc
        ON ind.PersonId = enc.PersonId
),
First_Step AS (
    SELECT
        *,

        -----------------------------------------------------------------------
        -- Event Time (valid death only)
        -----------------------------------------------------------------------
        CASE
            WHEN Event_Time IS NOT NULL
                 AND Event_Time > index_date
                 AND Event_Time <= Censor_Date
            THEN Event_Time
            ELSE NULL
        END AS Valid_Event_Time,

        CASE
            WHEN Event_Time IS NOT NULL
                 AND Event_Time > index_date
                 AND Event_Time <= Censor_Date
            THEN 'death'
            ELSE NULL
        END AS Event_Name,

        -----------------------------------------------------------------------
        -- Event or Censor Date
        -----------------------------------------------------------------------
        CASE
            -- Invalid
            WHEN (Event_Time IS NOT NULL AND Event_Time <= index_date)
                 OR index_date > DATE '2026-03-06'
            THEN NULL

            -- Event
            WHEN Event_Time IS NOT NULL
                 AND Event_Time > index_date
                 AND Event_Time <= Censor_Date
            THEN Event_Time

            -- Censored
            ELSE Censor_Date
        END AS Event_or_Censor_Date,

        -----------------------------------------------------------------------
        -- Outcome (1 = event, 0 = censored)
        -----------------------------------------------------------------------
        CASE
            WHEN Event_Time IS NOT NULL
                 AND Event_Time > index_date
                 AND Event_Time <= Censor_Date
            THEN 1
            ELSE 0
        END AS Outcome,

        -----------------------------------------------------------------------
        -- Outcome Text
        -----------------------------------------------------------------------
        CASE
            WHEN (Event_Time IS NOT NULL AND Event_Time <= index_date)
                 OR index_date > DATE '2026-03-06'
            THEN 'Invalid'

            WHEN Event_Time IS NOT NULL
                 AND Event_Time > index_date
                 AND Event_Time <= Censor_Date
            THEN 'Event'

            WHEN Last_Encounter_Date IS NOT NULL
                 AND Last_Encounter_Date > index_date
                 AND Censor_Date = DATEADD(year, 1, Last_Encounter_Date)
            THEN 'Censored by Encounter Gap (1yr)'

            WHEN Censor_Date = DATEADD(year, 3, index_date)
            THEN 'Censored at Max Follow-up (3yr)'

            ELSE 'Censored at Administrative Cutoff'
        END AS Outcome_Text

    FROM Censor_Calc
)
SELECT
    *,
    DATEDIFF(day, index_date, Event_or_Censor_Date) AS Survival_Time
FROM First_Step
ORDER BY PersonId
"""
death_outcome_time_status = spark.sql(death_outcome_time_status_query)
death_outcome_time_status.createOrReplaceTempView('tbl_death_outcome_time_status')

In [ ]:
death_outcome_time_status = death_outcome_time_status.drop_duplicates()
save_path = "/aim_1b/Secondary_Endpoint_All_cause_mortality_t2d_secondary_female"
study.save_artifacts_data(df=death_outcome_time_status, path=save_path, data_type="parquet", mode="overwrite")

# Primary Endpoint: Major Adverse Cardiovascular Events (MACE)


## 1. Myocardial infarction (MI)

In [ ]:
# Define AMI ICD codes
ami1 = ['I21', 'I22', 'I25.2']
ami2 = ['410', '412']
ami_icd10_codes = snapshot.codeset('ICD10CM', 'selfAndDescendants', *ami1, view_name='tbl_ami_icd10_code_set')
ami_icd9_codes = snapshot.codeset('ICD9CM', 'selfAndDescendants', *ami2, view_name='tbl_ami_icd9_code_set')

# Combine ICD-9 and ICD-10 codes
ami_combined_codes = ps.concat([ami_icd10_codes, ami_icd9_codes])
study.create_view(ami_combined_codes, view_name = 'tbl_combined_ami_code_set')

# Load AMI diagnoses filtered by codes
ami_combined_df = snapshot.load_filtered_table('Condition', ami_combined_codes, apply_annulled_filter=True, view_name='tbl_combined_ami_diagnosis')

# Select primary diagnoses for cohort patients
patient_ami_query = """
SELECT DISTINCT PersonId, PrimaryDiagnosisConceptId, RecordedDateTime
FROM tbl_combined_ami_diagnosis
WHERE PersonId IN (SELECT DISTINCT PersonId FROM tbl_patient_index_date) AND RecordedDateTime IS NOT NULL
AND VerificationStatusConceptId NOT IN (1065195, 1065198)
AND ClinicalStatusConceptId NOT IN (1065184, 1065178, 1065183)
AND CategoryConceptId IN (1065168, 1065169, 1065170)
"""
patient_ami = spark.sql(patient_ami_query)
patient_ami.createOrReplaceTempView('tbl_patient_ami')

# VerificationStatusConceptId: Exclude refuted, erroneous ones
# ClinicalStatusConceptId: Exclude resolved, inactive, remission ones (for only currently active in episodes?)
# CategoryConceptId: Only billing, encounter, admission diagnoses
## Primary Diagnosis Indicator Codes:
## 1200405: No
## 1200406: Yes
## 1067557: Field is not present in source
## 1067561: No Information

## 2. Stroke


In [ ]:
# Define Stroke ICD codes
stroke1 = ['I60', 'I61', 'I63']
stroke2 = ['430', '431', '432', '433.1', '434.1', '436']
stroke_icd10_codes = snapshot.codeset('ICD10CM', 'selfAndDescendants', *stroke1, view_name='tbl_stroke_icd10_code_set')
stroke_icd9_codes = snapshot.codeset('ICD9CM', 'selfAndDescendants', *stroke2, view_name='tbl_stroke_icd9_code_set')

# Combine ICD-9 and ICD-10 codes
stroke_combined_codes = ps.concat([stroke_icd10_codes, stroke_icd9_codes])
study.create_view(stroke_combined_codes, view_name = 'tbl_combined_stroke_code_set')

# Load stroke diagnoses filtered by codes
stroke_combined_df = snapshot.load_filtered_table('Condition', stroke_combined_codes, apply_annulled_filter=True, view_name='tbl_combined_stroke_diagnosis')

# Select primary diagnoses for cohort patients
patient_stroke_query = """
SELECT DISTINCT PersonId, PrimaryDiagnosisConceptId, RecordedDateTime
FROM tbl_combined_stroke_diagnosis
WHERE PersonId IN (SELECT DISTINCT PersonId FROM tbl_patient_index_date) AND RecordedDateTime IS NOT NULL
AND VerificationStatusConceptId NOT IN (1065195, 1065198)
AND ClinicalStatusConceptId NOT IN (1065184, 1065178, 1065183)
AND CategoryConceptId IN (1065168, 1065169, 1065170)
"""
patient_stroke = spark.sql(patient_stroke_query)
patient_stroke.createOrReplaceTempView('tbl_patient_stroke')

## 3. MACE Composite (First MI or Stroke)

In [ ]:
# Inpatient hospitalization encounter
inpatient_encounter_query = """
SELECT DISTINCT PersonId, StartDateTime, EndDateTime, ClassConceptId, TypeConceptId
FROM Encounter
WHERE PersonId IN (SELECT DISTINCT PersonId FROM tbl_patient_index_date)
AND StatusConceptId NOT IN (2506590, 2506591, 2983199, 2983200, 2506594, 2506595)
AND DischargeDispositionConceptId NOT IN (1065250)
AND StartDateTime IS NOT NULL
AND (EndDateTime IS NULL OR EndDateTime >= StartDateTime)
AND (
    ClassConceptId IN (
        1065603, 1065304, 1065307, 3059272, 1065297,
        1065302, 1065215, 1065220, 1065225, 1065227
    )
    OR TypeConceptId IN (
        1065603, 1065304, 1065307, 3059272, 1065297,
        1065302, 1065215, 1065220, 1065225, 1065227
    )
)
"""
inpatient_encounter = spark.sql(inpatient_encounter_query)
inpatient_encounter.createOrReplaceTempView("tbl_encounter_inpatient_hospitalization")

In [ ]:
# Urgent visit encounter
urgentvisit_encounter_query = """
SELECT DISTINCT PersonId, StartDateTime, EndDateTime, ClassConceptId, TypeConceptId
FROM Encounter
WHERE PersonId IN (SELECT DISTINCT PersonId FROM tbl_patient_index_date)
AND StatusConceptId NOT IN (2506590, 2506591, 2983199, 2983200, 2506594, 2506595)
AND DischargeDispositionConceptId NOT IN (1065250)
AND StartDateTime IS NOT NULL
AND (EndDateTime IS NULL OR EndDateTime >= StartDateTime)
AND (
    ClassConceptId IN (1065217, 1065290, 1065270)
    OR TypeConceptId IN (1065217, 1065290, 1065270)
)
"""
urgentvisit_encounter = spark.sql(urgentvisit_encounter_query)
urgentvisit_encounter.createOrReplaceTempView("tbl_encounter_urgentvisit")

In [ ]:
mi_condition_inpatient_encounter_query = """
SELECT e.*,
       i.index_date,
       d.RecordedDateTime
FROM tbl_encounter_inpatient_hospitalization e
INNER JOIN tbl_patient_ami d
  ON d.PersonId = e.PersonId
  AND d.RecordedDateTime BETWEEN e.StartDateTime AND DATEADD(day, 1, e.StartDateTime)
INNER JOIN tbl_patient_index_date i
  ON e.PersonId = i.PersonId
  AND e.StartDateTime > i.index_date
"""
mi_condition_inpatient_encounter = spark.sql(mi_condition_inpatient_encounter_query)
mi_condition_inpatient_encounter.createOrReplaceTempView('tbl_mi_condition_and_inpatient_encounter')

In [ ]:
mi_condition_urgentvisit_encounter_query = """
SELECT e.*,
       i.index_date,
       d.RecordedDateTime
FROM tbl_encounter_urgentvisit e
INNER JOIN tbl_patient_ami d
  ON d.PersonId = e.PersonId
  AND d.RecordedDateTime BETWEEN e.StartDateTime AND DATEADD(day, 1, e.StartDateTime)
INNER JOIN tbl_patient_index_date i
  ON e.PersonId = i.PersonId
  AND e.StartDateTime > i.index_date
"""
mi_condition_urgentvisit_encounter = spark.sql(mi_condition_urgentvisit_encounter_query)
mi_condition_urgentvisit_encounter.createOrReplaceTempView('tbl_mi_condition_and_urgentvisit_encounter')

In [ ]:
mi_event_union_query = """
SELECT *
FROM tbl_mi_condition_and_inpatient_encounter a
UNION
SELECT *
FROM tbl_mi_condition_and_urgentvisit_encounter b
"""
mi_event_union = spark.sql(mi_event_union_query)
mi_event_union.createOrReplaceTempView("tbl_mi_event_union")

In [ ]:
first_mi_event_query = """
WITH Ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PersonId, index_date
            ORDER BY RecordedDateTime ASC
        ) AS rn
    FROM tbl_mi_event_union
)
SELECT
    PersonId,
    index_date,
    RecordedDateTime AS First_Mi_Time
FROM Ranked
WHERE rn = 1;
"""
first_mi_event = spark.sql(first_mi_event_query)
first_mi_event.createOrReplaceTempView('tbl_first_mi_event')

In [ ]:
stroke_condition_inpatient_encounter_query = """
SELECT e.*,
       i.index_date,
       d.RecordedDateTime
FROM tbl_encounter_inpatient_hospitalization e
INNER JOIN tbl_patient_stroke d
  ON d.PersonId = e.PersonId
  AND d.RecordedDateTime BETWEEN e.StartDateTime AND DATEADD(day, 1, e.StartDateTime)
INNER JOIN tbl_patient_index_date i
  ON e.PersonId = i.PersonId
  AND e.StartDateTime > i.index_date
"""
stroke_condition_inpatient_encounter = spark.sql(stroke_condition_inpatient_encounter_query)
stroke_condition_inpatient_encounter.createOrReplaceTempView('tbl_stroke_condition_and_inpatient_encounter')

In [ ]:
stroke_condition_urgentvisit_encounter_query = """
SELECT e.*,
       i.index_date,
       d.RecordedDateTime
FROM tbl_encounter_urgentvisit e
INNER JOIN tbl_patient_stroke d
  ON d.PersonId = e.PersonId
  AND d.RecordedDateTime BETWEEN e.StartDateTime AND DATEADD(day, 1, e.StartDateTime)
INNER JOIN tbl_patient_index_date i
  ON e.PersonId = i.PersonId
  AND e.StartDateTime > i.index_date
"""
stroke_condition_urgentvisit_encounter = spark.sql(stroke_condition_urgentvisit_encounter_query)
stroke_condition_urgentvisit_encounter.createOrReplaceTempView('tbl_stroke_condition_and_urgentvisit_encounter')

In [ ]:
stroke_event_union_query = """
SELECT *
FROM tbl_stroke_condition_and_inpatient_encounter a
UNION
SELECT *
FROM tbl_stroke_condition_and_urgentvisit_encounter b
"""
stroke_event_union = spark.sql(stroke_event_union_query)
stroke_event_union.createOrReplaceTempView("tbl_stroke_event_union")

In [ ]:
first_stroke_event_query = """
WITH Ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PersonId, index_date
            ORDER BY RecordedDateTime ASC
        ) AS rn
    FROM tbl_stroke_event_union
)
SELECT
    PersonId,
    index_date,
    RecordedDateTime AS First_Stroke_Time
FROM Ranked
WHERE rn = 1;
"""
first_stroke_event = spark.sql(first_stroke_event_query)
first_stroke_event.createOrReplaceTempView('tbl_first_stroke_event')

In [ ]:
first_mi_stroke_event_union_query = """
SELECT a.PersonId,
       a.index_date,
       "MI" AS Event_Name,
       a.First_Mi_Time AS Event_Time
FROM tbl_first_mi_event a
UNION
SELECT b.PersonId,
       b.index_date,
       "Stroke" AS Event_Name,
       b.First_Stroke_Time AS Event_Time
FROM tbl_first_stroke_event b
"""
first_mi_stroke_event_union = spark.sql(first_mi_stroke_event_union_query)
first_mi_stroke_event_union.createOrReplaceTempView("tbl_first_mi_stroke_event_union")

In [ ]:
first_selected_event_query = """
WITH RankedEvents AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY PersonId, index_date ORDER BY Event_Time ASC) AS id
    FROM tbl_first_mi_stroke_event_union)
SELECT
    PersonId,
    index_date,
    Event_Time,
    Event_Name
FROM RankedEvents
WHERE id = 1
"""
first_selected_event = spark.sql(first_selected_event_query)
first_selected_event.createOrReplaceTempView("tbl_first_selected_event")

In [ ]:
# Encounter
encounter_query = """
SELECT
PersonId, StartDateTime, EndDateTime, ClassConceptId
FROM Encounter
WHERE PersonId IN (SELECT DISTINCT PersonId FROM tbl_patient_index_date)
AND StatusConceptId NOT IN (2506590, 2506591, 2983199, 2983200, 2506594, 2506595)
AND DischargeDispositionConceptId NOT IN (1065250)
AND StartDateTime IS NOT NULL
AND (EndDateTime IS NULL OR EndDateTime >= StartDateTime)

"""
encounter = spark.sql(encounter_query)
encounter.createOrReplaceTempView("tbl_encounter")

In [ ]:
# Last Encounter
last_encounter_query = """
SELECT
PersonId, MAX(EndDateTime) AS Last_Encounter_Date
FROM tbl_encounter
GROUP BY 1
"""
last_encounter = spark.sql(last_encounter_query)
last_encounter.createOrReplaceTempView("tbl_last_encounter")

In [ ]:
ehr_union_outcome_time_status_query = """
WITH Censor_Calc AS (
    SELECT
        ind.PersonId,
        ind.index_date,
        event.Event_Time,
        event.Event_Name,
        d.Death_Time,
        enc.Last_Encounter_Date,
        LEAST(
            DATE '2026-03-06',
            COALESCE(d.Death_Time, DATE '2026-03-06'),
            CASE
                WHEN enc.Last_Encounter_Date >= ind.index_date
                THEN DATEADD(year, 1, enc.Last_Encounter_Date)
                ELSE DATE '2026-03-06'
            END,
            DATEADD(year, 3, ind.index_date)
        ) AS Censor_Date

    FROM tbl_patient_index_date ind
    LEFT JOIN tbl_first_selected_event event ON ind.PersonId = event.PersonId
    LEFT JOIN tbl_death d                    ON ind.PersonId = d.PersonId
    LEFT JOIN tbl_last_encounter enc         ON ind.PersonId = enc.PersonId
),
First_Step AS (
    SELECT
        *,
        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN NULL
            WHEN Event_Time IS NOT NULL
                 AND index_date < Event_Time
                 AND Event_Time <= Censor_Date
            THEN Event_Time
            ELSE Censor_Date
        END AS Event_or_Censor_Date,

        CASE
            WHEN Event_Time IS NOT NULL
                 AND index_date < Event_Time
                 AND Event_Time <= Censor_Date
            THEN 1
            ELSE 0
        END AS Outcome,

        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN 'Invalid'
            WHEN Event_Time IS NOT NULL
                 AND index_date < Event_Time
                 AND Event_Time <= Censor_Date
            THEN 'Event'
            WHEN Death_Time IS NOT NULL
                 AND Censor_Date = Death_Time
            THEN 'Censored by Death'
            WHEN Last_Encounter_Date IS NOT NULL
                 AND Last_Encounter_Date >= index_date
                 AND Censor_Date = DATEADD(year, 1, Last_Encounter_Date)
            THEN 'Censored by Encounter Gap (1yr)'
            WHEN Censor_Date = DATEADD(year, 3, index_date)
            THEN 'Censored at Max Follow-up (3yr)'
            ELSE 'Censored at Administrative Cutoff'
        END AS Outcome_Text

    FROM Censor_Calc
)
SELECT
    *,
    DATEDIFF(day, index_date, Event_or_Censor_Date) AS Survival_Time
FROM First_Step
ORDER BY PersonId
"""
ehr_union_outcome_time_status = spark.sql(ehr_union_outcome_time_status_query)
ehr_union_outcome_time_status.createOrReplaceTempView('tbl_ehr_union_primary_outcome_time_status')

In [ ]:
ehr_union_outcome_time_status = ehr_union_outcome_time_status.drop_duplicates()
save_path = "/aim_1b/Primary_Endpoint_t2d_secondary_female"
study.save_artifacts_data(df=ehr_union_outcome_time_status, path=save_path, data_type="parquet", mode="overwrite")

## 4. MACE + Heart Failure Hospitalization (Expanded Composite)

In [ ]:
hf_icd9 = ['398.91', '402.01', '402.11', '402.91', '404.01', '404.03', '404.11', '404.13',
           '404.91', '404.93', '425.4', '425.5', '425.6', '425.7', '425.8', '425.9', '428']
hf_icd10 = ['I09.9', 'I11.0', 'I13.0', 'I13.2', 'I25.5', 'I42.0', 'I42.5', 'I42.6',
         'I42.7', 'I42.8', 'I42.9', 'I43', 'I50', 'P29.0']
hf_icd9_codeset = snapshot.codeset('ICD9CM', 'selfAndDescendants', *hf_icd9, view_name = "tbl_hf_icd9")
hf_icd10_codeset = snapshot.codeset('ICD10CM', 'selfAndDescendants', *hf_icd10, view_name = "tbl_hf_icd10")
hf_codeset = ps.concat([hf_icd9_codeset, hf_icd10_codeset])
study.create_view(hf_codeset, view_name = 'tbl_combined_hf_code_set')
hf_condition = snapshot.load_filtered_table('Condition', hf_codeset, apply_annulled_filter=True, view_name = "tbl_hf_diagnosis")

patient_hf_query = """
SELECT DISTINCT PersonId, PrimaryDiagnosisConceptId, RecordedDateTime
FROM tbl_hf_diagnosis
WHERE PersonId IN (SELECT DISTINCT PersonId FROM tbl_patient_index_date) AND RecordedDateTime IS NOT NULL
AND VerificationStatusConceptId NOT IN (1065195, 1065198)
AND ClinicalStatusConceptId NOT IN (1065184, 1065178, 1065183)
AND CategoryConceptId IN (1065168, 1065169, 1065170)
"""
patient_hf = spark.sql(patient_hf_query)
patient_hf.createOrReplaceTempView('tbl_patient_hf')

In [ ]:
hf_condition_inpatient_encounter_query = """
SELECT e.*,
       i.index_date,
       d.RecordedDateTime
FROM tbl_encounter_inpatient_hospitalization e
INNER JOIN tbl_patient_hf d
  ON d.PersonId = e.PersonId
  AND d.RecordedDateTime BETWEEN e.StartDateTime AND DATEADD(day, 1, e.StartDateTime)
INNER JOIN tbl_patient_index_date i
  ON e.PersonId = i.PersonId
  AND e.StartDateTime > i.index_date
"""
hf_condition_inpatient_encounter = spark.sql(hf_condition_inpatient_encounter_query)
hf_condition_inpatient_encounter.createOrReplaceTempView('tbl_hf_condition_and_inpatient_encounter')

In [ ]:
first_hf_event_query = """
WITH Ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PersonId, index_date
            ORDER BY RecordedDateTime ASC
        ) AS rn
    FROM tbl_hf_condition_and_inpatient_encounter
)
SELECT
    PersonId,
    index_date,
    RecordedDateTime AS First_Hf_Time
FROM Ranked
WHERE rn = 1;
"""
first_hf_event = spark.sql(first_hf_event_query)
first_hf_event.createOrReplaceTempView('tbl_first_hf_event')

In [ ]:
first_mi_stroke_hf_event_union_query = """
SELECT a.PersonId,
       a.index_date,
       "MI" AS Event_Name,
       a.First_Mi_Time AS Event_Time
FROM tbl_first_mi_event a
UNION
SELECT b.PersonId,
       b.index_date,
       "Stroke" AS Event_Name,
       b.First_Stroke_Time AS Event_Time
FROM tbl_first_stroke_event b
UNION
SELECT c.PersonId,
       c.index_date,
       "Heart Failure" AS Event_Name,
       c.First_Hf_Time AS Event_Time
FROM tbl_first_hf_event c
"""
first_mi_stroke_hf_event_union = spark.sql(first_mi_stroke_hf_event_union_query)
first_mi_stroke_hf_event_union.createOrReplaceTempView("tbl_first_mi_stroke_hf_event_union")

In [ ]:
first_selected_event_query_s = """
WITH RankedEvents AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY PersonId, index_date ORDER BY Event_Time ASC) AS id
    FROM tbl_first_mi_stroke_hf_event_union)
SELECT
    PersonId,
    index_date,
    Event_Time,
    Event_Name
FROM RankedEvents
WHERE id = 1
"""
first_selected_event_s = spark.sql(first_selected_event_query_s)
first_selected_event_s.createOrReplaceTempView("tbl_first_selected_event_sec")

In [ ]:
first_sec_union_outcome_time_status_query = """
WITH Censor_Calc AS (
    SELECT
        ind.PersonId,
        ind.index_date,
        event.Event_Time,
        event.Event_Name,
        d.Death_Time,
        enc.Last_Encounter_Date,
        LEAST(
            DATE '2026-03-06',
            COALESCE(d.Death_Time, DATE '2026-03-06'),
            CASE
                WHEN enc.Last_Encounter_Date >= ind.index_date
                THEN DATEADD(year, 1, enc.Last_Encounter_Date)
                ELSE DATE '2026-03-06'
            END,
            DATEADD(year, 3, ind.index_date)
        ) AS Censor_Date

    FROM tbl_patient_index_date ind
    LEFT JOIN tbl_first_selected_event_sec event ON ind.PersonId = event.PersonId
    LEFT JOIN tbl_death d                         ON ind.PersonId = d.PersonId
    LEFT JOIN tbl_last_encounter enc              ON ind.PersonId = enc.PersonId
),
First_Step AS (
    SELECT
        *,
        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN NULL
            WHEN Event_Time IS NOT NULL
                 AND index_date < Event_Time
                 AND Event_Time <= Censor_Date
            THEN Event_Time
            ELSE Censor_Date
        END AS Event_or_Censor_Date,

        CASE
            WHEN Event_Time IS NOT NULL
                 AND index_date < Event_Time
                 AND Event_Time <= Censor_Date
            THEN 1
            ELSE 0
        END AS Outcome,

        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN 'Invalid'
            WHEN Event_Time IS NOT NULL
                 AND index_date < Event_Time
                 AND Event_Time <= Censor_Date
            THEN 'Event'
            WHEN Death_Time IS NOT NULL
                 AND Censor_Date = Death_Time
            THEN 'Censored by Death'
            WHEN Last_Encounter_Date IS NOT NULL
                 AND Last_Encounter_Date >= index_date
                 AND Censor_Date = DATEADD(year, 1, Last_Encounter_Date)
            THEN 'Censored by Encounter Gap (1yr)'
            WHEN Censor_Date = DATEADD(year, 3, index_date)
            THEN 'Censored at Max Follow-up (3yr)'
            ELSE 'Censored at Administrative Cutoff'
        END AS Outcome_Text

    FROM Censor_Calc
)
SELECT
    *,
    DATEDIFF(day, index_date, Event_or_Censor_Date) AS Survival_Time
FROM First_Step
ORDER BY PersonId
"""
first_sec_union_outcome_time_status = spark.sql(first_sec_union_outcome_time_status_query)
first_sec_union_outcome_time_status.createOrReplaceTempView('tbl_first_sec_union_outcome_time_status')

In [ ]:
first_sec_union_outcome_time_status = first_sec_union_outcome_time_status.drop_duplicates()
save_path = "/aim_1b/Secondary_Endpoint_add_HF_hospitalization_t2d_secondary_female"
study.save_artifacts_data(df=first_sec_union_outcome_time_status, path=save_path, data_type="parquet", mode="overwrite")

# Secondary Endpoints

## 1. CKD (Stage 3-5)

In [ ]:
ICD9_CKD35 = ['585', '403', '404']
ICD10_CKD35 = ['N18', 'I12', 'I13']
ICD9_CKD35_codeset = snapshot.codeset('ICD9CM', 'selfAndDescendants', *ICD9_CKD35)
ICD10_CKD35_codeset = snapshot.codeset('ICD10CM', 'selfAndDescendants', *ICD10_CKD35)
CKD35_codeset = ps.concat([ICD9_CKD35_codeset, ICD10_CKD35_codeset])
CKD35 = snapshot.load_filtered_table('Condition', CKD35_codeset, apply_annulled_filter=True, view_name = "tbl_CKD35_diagnosis")

In [ ]:
first_ckd35_diagnosis_query = """
WITH Ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PersonId
            ORDER BY RecordedDateTime ASC
        ) AS rn
    FROM tbl_CKD35_diagnosis
)
SELECT
    PersonId,
    RecordedDateTime
FROM Ranked
WHERE rn = 1;
"""
first_ckd35_event = spark.sql(first_ckd35_diagnosis_query)
first_ckd35_event.createOrReplaceTempView('tbl_first_ckd35_diagnosis')

In [ ]:
CKD35_history_during_lookback_query = """
SELECT
    i.PersonId,
    COUNT(c.RecordedDateTime) AS CKD35_history
FROM tbl_patient_index_date i
LEFT JOIN tbl_CKD35_diagnosis c ON i.PersonId = c.PersonId AND c.RecordedDateTime >= i.look_back AND c.RecordedDateTime <= i.index_date
WHERE c.RecordedDateTime IS NOT NULL
GROUP BY i.PersonId
"""
CKD35_history_during_lookback = spark.sql(CKD35_history_during_lookback_query)
CKD35_history_during_lookback.createOrReplaceTempView("tbl_CKD35_history")

In [ ]:
ckd_outcome_time_status_query = """
WITH Censor_Calc AS (
    SELECT
        ind.PersonId,
        ind.index_date,
        event.RecordedDateTime,
        d.Death_Time,
        enc.Last_Encounter_Date,
        LEAST(
            DATE '2026-03-06',                                                      -- Administrative cutoff
            COALESCE(d.Death_Time,                          DATE '2026-03-06'),     -- Death
            CASE
                WHEN enc.Last_Encounter_Date >= ind.index_date
                THEN DATEADD(year, 1, enc.Last_Encounter_Date)
                ELSE DATE '2026-03-06'
            END,
            DATEADD(year, 3, ind.index_date)                                        -- 3yr max follow-up
        ) AS Censor_Date

    FROM tbl_patient_index_date ind
    LEFT JOIN tbl_first_ckd35_diagnosis event
        ON ind.PersonId = event.PersonId AND event.RecordedDateTime > ind.index_date
    LEFT JOIN tbl_death d
        ON ind.PersonId = d.PersonId
    LEFT JOIN tbl_last_encounter enc
        ON ind.PersonId = enc.PersonId
    WHERE ind.PersonId NOT IN (
        SELECT PersonId
        FROM tbl_CKD35_history
        WHERE CKD35_history > 0
    )
),
First_Step AS (
    SELECT
        *,

        -----------------------------------------------------------------------
        -- Event or Censor Date
        -----------------------------------------------------------------------
        CASE
            -- Invalid
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN NULL

            -- Event
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN RecordedDateTime

            -- Censored
            ELSE Censor_Date
        END AS Event_or_Censor_Date,

        -----------------------------------------------------------------------
        -- Outcome (1 = event, 0 = censored)
        -----------------------------------------------------------------------
        CASE
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 1
            ELSE 0
        END AS Outcome,

        -----------------------------------------------------------------------
        -- Outcome Text
        -----------------------------------------------------------------------
        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN 'Invalid'

            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 'Event'

            WHEN Death_Time IS NOT NULL
                 AND Censor_Date = Death_Time
            THEN 'Censored by Death'

            WHEN Last_Encounter_Date IS NOT NULL
                 AND Last_Encounter_Date >= index_date
                 AND Censor_Date = DATEADD(year, 1, Last_Encounter_Date)
            THEN 'Censored by Encounter Gap (1yr)'

            WHEN Censor_Date = DATEADD(year, 3, index_date)
            THEN 'Censored at Max Follow-up (3yr)'

            ELSE 'Censored at Administrative Cutoff'
        END AS Outcome_Text

    FROM Censor_Calc
)
SELECT
    *,
    DATEDIFF(day, index_date, Event_or_Censor_Date) AS Survival_Time
FROM First_Step
ORDER BY PersonId
"""
ckd_outcome_time_status = spark.sql(ckd_outcome_time_status_query)
ckd_outcome_time_status.createOrReplaceTempView("tbl_CKD35_Endpoint")

In [ ]:
ckd_outcome_time_status = ckd_outcome_time_status.drop_duplicates()
save_path = "/aim_1b/CKD_Endpoint_t2d_secondary_female"
study.save_artifacts_data(df=ckd_outcome_time_status, path=save_path, data_type="parquet", mode="overwrite")

## 2. MASLD

In [ ]:
ICD9_MASLD = ['571.8', '571.9']
ICD10_MASLD = ['K75.8', 'K76.0']
ICD9_MASLD_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_MASLD)
ICD10_MASLD_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_MASLD)
MASLD_codeset = ps.concat([ICD9_MASLD_codeset, ICD10_MASLD_codeset])
MASLD_condition = snapshot.load_filtered_table('Condition', MASLD_codeset, apply_annulled_filter=True, view_name = "tbl_MASLD_diagnosis")

In [ ]:
first_masld_diagnosis_query = """
WITH Ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PersonId
            ORDER BY RecordedDateTime ASC
        ) AS rn
    FROM tbl_MASLD_diagnosis
)
SELECT
    PersonId,
    RecordedDateTime
FROM Ranked
WHERE rn = 1;
"""
first_masld_event = spark.sql(first_masld_diagnosis_query)
first_masld_event.createOrReplaceTempView('tbl_first_masld_diagnosis')

In [ ]:
MASLD_history_during_lookback_query = """
SELECT
    i.PersonId,
    COUNT(c.RecordedDateTime) AS MASLD_history
FROM tbl_patient_index_date i
LEFT JOIN tbl_MASLD_diagnosis c ON i.PersonId = c.PersonId AND c.RecordedDateTime >= i.look_back AND c.RecordedDateTime <= i.index_date
WHERE c.RecordedDateTime IS NOT NULL
GROUP BY i.PersonId
"""
MASLD_history_during_lookback = spark.sql(MASLD_history_during_lookback_query)
MASLD_history_during_lookback.createOrReplaceTempView("tbl_MASLD_history")

In [ ]:
masld_outcome_time_status_query = """
WITH Censor_Calc AS (
    SELECT
        ind.PersonId,
        ind.index_date,
        event.RecordedDateTime,
        d.Death_Time,
        enc.Last_Encounter_Date,
        LEAST(
            DATE '2026-03-06',                                                      -- Administrative cutoff
            COALESCE(d.Death_Time,                          DATE '2026-03-06'),     -- Death
            CASE
                WHEN enc.Last_Encounter_Date >= ind.index_date
                THEN DATEADD(year, 1, enc.Last_Encounter_Date)
                ELSE DATE '2026-03-06'
            END,
            DATEADD(year, 3, ind.index_date)                                        -- 3yr max follow-up
        ) AS Censor_Date

    FROM tbl_patient_index_date ind
    LEFT JOIN tbl_first_masld_diagnosis event
        ON ind.PersonId = event.PersonId AND event.RecordedDateTime > ind.index_date
    LEFT JOIN tbl_death d
        ON ind.PersonId = d.PersonId
    LEFT JOIN tbl_last_encounter enc
        ON ind.PersonId = enc.PersonId
    WHERE ind.PersonId NOT IN (
        SELECT PersonId
        FROM tbl_MASLD_history
        WHERE MASLD_history > 0
    )
),
First_Step AS (
    SELECT
        *,

        -----------------------------------------------------------------------
        -- Event or Censor Date
        -----------------------------------------------------------------------
        CASE
            -- Invalid
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN NULL

            -- Event
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN RecordedDateTime

            -- Censored
            ELSE Censor_Date
        END AS Event_or_Censor_Date,

        -----------------------------------------------------------------------
        -- Outcome (1 = event, 0 = censored)
        -----------------------------------------------------------------------
        CASE
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 1
            ELSE 0
        END AS Outcome,

        -----------------------------------------------------------------------
        -- Outcome Text
        -----------------------------------------------------------------------
        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN 'Invalid'

            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 'Event'

            WHEN Death_Time IS NOT NULL
                 AND Censor_Date = Death_Time
            THEN 'Censored by Death'

            WHEN Last_Encounter_Date IS NOT NULL
                 AND Last_Encounter_Date >= index_date
                 AND Censor_Date = DATEADD(year, 1, Last_Encounter_Date)
            THEN 'Censored by Encounter Gap (1yr)'

            WHEN Censor_Date = DATEADD(year, 3, index_date)
            THEN 'Censored at Max Follow-up (3yr)'

            ELSE 'Censored at Administrative Cutoff'
        END AS Outcome_Text

    FROM Censor_Calc
)
SELECT
    *,
    DATEDIFF(day, index_date, Event_or_Censor_Date) AS Survival_Time
FROM First_Step
ORDER BY PersonId
"""
masld_outcome_time_status = spark.sql(masld_outcome_time_status_query)
masld_outcome_time_status.createOrReplaceTempView("tbl_MASLD_Endpoint")

In [ ]:
masld_outcome_time_status = masld_outcome_time_status.drop_duplicates()
save_path = "/aim_1b/MASLD_Endpoint_t2d_secondary_female"
study.save_artifacts_data(df=masld_outcome_time_status, path=save_path, data_type="parquet", mode="overwrite")

# Negative control outcomes

## 1. Lumbar radiculopathy

In [ ]:
ICD9_lr = ['724.3']
ICD10_lr = ['M54.16', 'M54.17', 'M51.16', 'M47.26']
ICD9_lr_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_lr)
ICD10_lr_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_lr)
lr_codeset = ps.concat([ICD9_lr_codeset, ICD10_lr_codeset])
lr_condition = snapshot.load_filtered_table('Condition', lr_codeset, apply_annulled_filter=True, view_name = "tbl_lr_diagnosis")

In [ ]:
first_lr_diagnosis_query = """
WITH Ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PersonId
            ORDER BY RecordedDateTime ASC
        ) AS rn
    FROM tbl_lr_diagnosis
)
SELECT
    PersonId,
    RecordedDateTime
FROM Ranked
WHERE rn = 1;
"""
first_lr_event = spark.sql(first_lr_diagnosis_query)
first_lr_event.createOrReplaceTempView('tbl_first_lr_diagnosis')

In [ ]:
lr_history_during_lookback_query = """
SELECT
    i.PersonId,
    COUNT(c.RecordedDateTime) AS lr_history
FROM tbl_patient_index_date i
LEFT JOIN tbl_lr_diagnosis c ON i.PersonId = c.PersonId AND c.RecordedDateTime >= i.look_back AND c.RecordedDateTime <= i.index_date
WHERE c.RecordedDateTime IS NOT NULL
GROUP BY i.PersonId
"""
lr_history_during_lookback = spark.sql(lr_history_during_lookback_query)
lr_history_during_lookback.createOrReplaceTempView("tbl_lr_history")

In [ ]:
lr_outcome_time_status_query = """
WITH Censor_Calc AS (
    SELECT
        ind.PersonId,
        ind.index_date,
        event.RecordedDateTime,
        d.Death_Time,
        enc.Last_Encounter_Date,
        LEAST(
            DATE '2026-03-06',                                                      -- Administrative cutoff
            COALESCE(d.Death_Time,                          DATE '2026-03-06'),     -- Death
            CASE
                WHEN enc.Last_Encounter_Date >= ind.index_date
                THEN DATEADD(year, 1, enc.Last_Encounter_Date)
                ELSE DATE '2026-03-06'
            END,
            DATEADD(year, 3, ind.index_date)                                        -- 3yr max follow-up
        ) AS Censor_Date

    FROM tbl_patient_index_date ind
    LEFT JOIN tbl_first_lr_diagnosis event
        ON ind.PersonId = event.PersonId AND event.RecordedDateTime > ind.index_date
    LEFT JOIN tbl_death d
        ON ind.PersonId = d.PersonId
    LEFT JOIN tbl_last_encounter enc
        ON ind.PersonId = enc.PersonId
    WHERE ind.PersonId NOT IN (
        SELECT PersonId
        FROM tbl_lr_history
        WHERE lr_history > 0
    )
),
First_Step AS (
    SELECT
        *,

        -----------------------------------------------------------------------
        -- Event or Censor Date
        -----------------------------------------------------------------------
        CASE
            -- Invalid
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN NULL

            -- Event
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN RecordedDateTime

            -- Censored
            ELSE Censor_Date
        END AS Event_or_Censor_Date,

        -----------------------------------------------------------------------
        -- Outcome (1 = event, 0 = censored)
        -----------------------------------------------------------------------
        CASE
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 1
            ELSE 0
        END AS Outcome,

        -----------------------------------------------------------------------
        -- Outcome Text
        -----------------------------------------------------------------------
        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN 'Invalid'

            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 'Event'

            WHEN Death_Time IS NOT NULL
                 AND Censor_Date = Death_Time
            THEN 'Censored by Death'

            WHEN Last_Encounter_Date IS NOT NULL
                 AND Last_Encounter_Date >= index_date
                 AND Censor_Date = DATEADD(year, 1, Last_Encounter_Date)
            THEN 'Censored by Encounter Gap (1yr)'

            WHEN Censor_Date = DATEADD(year, 3, index_date)
            THEN 'Censored at Max Follow-up (3yr)'

            ELSE 'Censored at Administrative Cutoff'
        END AS Outcome_Text

    FROM Censor_Calc
)
SELECT
    *,
    DATEDIFF(day, index_date, Event_or_Censor_Date) AS Survival_Time
FROM First_Step
ORDER BY PersonId
"""
lr_outcome_time_status = spark.sql(lr_outcome_time_status_query)
lr_outcome_time_status.createOrReplaceTempView("tbl_lr_Endpoint")

In [ ]:
lr_outcome_time_status = lr_outcome_time_status.drop_duplicates()
save_path = "/aim_1b/lr_Endpoint_t2d_secondary_female"
study.save_artifacts_data(df=lr_outcome_time_status, path=save_path, data_type="parquet", mode="overwrite")

## 2. Abdominal hernia

In [ ]:
ICD9_ah = ['550', '551', '552', '553']
ICD10_ah = ['K40', 'K41', 'K42', 'K43', 'K45', 'K46']
ICD9_ah_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_ah)
ICD10_ah_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_ah)
ah_codeset = ps.concat([ICD9_ah_codeset, ICD10_ah_codeset])
ah_condition = snapshot.load_filtered_table('Condition', ah_codeset, apply_annulled_filter=True, view_name = "tbl_ah_diagnosis")

In [ ]:
first_ah_diagnosis_query = """
WITH Ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PersonId
            ORDER BY RecordedDateTime ASC
        ) AS rn
    FROM tbl_ah_diagnosis
)
SELECT
    PersonId,
    RecordedDateTime
FROM Ranked
WHERE rn = 1;
"""
first_ah_event = spark.sql(first_ah_diagnosis_query)
first_ah_event.createOrReplaceTempView('tbl_first_ah_diagnosis')

In [ ]:
ah_history_during_lookback_query = """
SELECT
    i.PersonId,
    COUNT(c.RecordedDateTime) AS ah_history
FROM tbl_patient_index_date i
LEFT JOIN tbl_ah_diagnosis c ON i.PersonId = c.PersonId AND c.RecordedDateTime >= i.look_back AND c.RecordedDateTime <= i.index_date
WHERE c.RecordedDateTime IS NOT NULL
GROUP BY i.PersonId
"""
ah_history_during_lookback = spark.sql(ah_history_during_lookback_query)
ah_history_during_lookback.createOrReplaceTempView("tbl_ah_history")

In [ ]:
ah_outcome_time_status_query = """
WITH Censor_Calc AS (
    SELECT
        ind.PersonId,
        ind.index_date,
        event.RecordedDateTime,
        d.Death_Time,
        enc.Last_Encounter_Date,
        LEAST(
            DATE '2026-03-06',                                                      -- Administrative cutoff
            COALESCE(d.Death_Time,                          DATE '2026-03-06'),     -- Death
            CASE
                WHEN enc.Last_Encounter_Date >= ind.index_date
                THEN DATEADD(year, 1, enc.Last_Encounter_Date)
                ELSE DATE '2026-03-06'
            END,
            DATEADD(year, 3, ind.index_date)                                        -- 3yr max follow-up
        ) AS Censor_Date

    FROM tbl_patient_index_date ind
    LEFT JOIN tbl_first_ah_diagnosis event
        ON ind.PersonId = event.PersonId AND event.RecordedDateTime > ind.index_date
    LEFT JOIN tbl_death d
        ON ind.PersonId = d.PersonId
    LEFT JOIN tbl_last_encounter enc
        ON ind.PersonId = enc.PersonId
    WHERE ind.PersonId NOT IN (
        SELECT PersonId
        FROM tbl_ah_history
        WHERE ah_history > 0
    )
),
First_Step AS (
    SELECT
        *,

        -----------------------------------------------------------------------
        -- Event or Censor Date
        -----------------------------------------------------------------------
        CASE
            -- Invalid
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN NULL

            -- Event
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN RecordedDateTime

            -- Censored
            ELSE Censor_Date
        END AS Event_or_Censor_Date,

        -----------------------------------------------------------------------
        -- Outcome (1 = event, 0 = censored)
        -----------------------------------------------------------------------
        CASE
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 1
            ELSE 0
        END AS Outcome,

        -----------------------------------------------------------------------
        -- Outcome Text
        -----------------------------------------------------------------------
        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN 'Invalid'

            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 'Event'

            WHEN Death_Time IS NOT NULL
                 AND Censor_Date = Death_Time
            THEN 'Censored by Death'

            WHEN Last_Encounter_Date IS NOT NULL
                 AND Last_Encounter_Date >= index_date
                 AND Censor_Date = DATEADD(year, 1, Last_Encounter_Date)
            THEN 'Censored by Encounter Gap (1yr)'

            WHEN Censor_Date = DATEADD(year, 3, index_date)
            THEN 'Censored at Max Follow-up (3yr)'

            ELSE 'Censored at Administrative Cutoff'
        END AS Outcome_Text

    FROM Censor_Calc
)
SELECT
    *,
    DATEDIFF(day, index_date, Event_or_Censor_Date) AS Survival_Time
FROM First_Step
ORDER BY PersonId
"""
ah_outcome_time_status = spark.sql(ah_outcome_time_status_query)
ah_outcome_time_status.createOrReplaceTempView("tbl_ah_Endpoint")

In [ ]:
ah_outcome_time_status = ah_outcome_time_status.drop_duplicates()
save_path = "/aim_1b/ah_Endpoint_t2d_secondary_female"
study.save_artifacts_data(df=ah_outcome_time_status, path=save_path, data_type="parquet", mode="overwrite")

# Harm endpoints

## Pancreatitis

In [ ]:
ICD9_all_pancreatitis = ['577.0', '577.1']
ICD10_all_pancreatitis = ['K85', 'K85.0', 'K85.1', 'K85.2', 'K85.3', 'K85.8', 'K85.9', 'K86.0', 'K86.1']
ICD9_all_pancreatitis_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_all_pancreatitis)
ICD10_all_pancreatitis_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_all_pancreatitis)
all_pancreatitis_codeset = ps.concat([ICD9_all_pancreatitis_codeset, ICD10_all_pancreatitis_codeset])
all_pancreatitis_condition = snapshot.load_filtered_table('Condition', all_pancreatitis_codeset, apply_annulled_filter=True, view_name = "tbl_all_pancreatitis_diagnosis")

In [ ]:
ICD9_acute_pancreatitis = ['577.0']
ICD10_acute_pancreatitis = ['K85', 'K85.0', 'K85.1', 'K85.2', 'K85.3', 'K85.8', 'K85.9']
ICD9_acute_pancreatitis_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_acute_pancreatitis)
ICD10_acute_pancreatitis_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_acute_pancreatitis)
acute_pancreatitis_codeset = ps.concat([ICD9_acute_pancreatitis_codeset, ICD10_acute_pancreatitis_codeset])
acute_pancreatitis_condition = snapshot.load_filtered_table('Condition', acute_pancreatitis_codeset, apply_annulled_filter=True, view_name = "tbl_acute_pancreatitis_diagnosis")

In [ ]:
first_pancreatitis_diagnosis_query = """
WITH Ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PersonId
            ORDER BY RecordedDateTime ASC
        ) AS rn
    FROM tbl_acute_pancreatitis_diagnosis
)
SELECT
    PersonId,
    RecordedDateTime
FROM Ranked
WHERE rn = 1;
"""
first_pancreatitis_event = spark.sql(first_pancreatitis_diagnosis_query)
first_pancreatitis_event.createOrReplaceTempView('tbl_first_pancreatitis_diagnosis')

In [ ]:
pancreatitis_history_during_lookback_query = """
SELECT
    i.PersonId,
    COUNT(c.RecordedDateTime) AS pancreatitis_history
FROM tbl_patient_index_date i
LEFT JOIN tbl_all_pancreatitis_diagnosis c ON i.PersonId = c.PersonId AND c.RecordedDateTime >= i.look_back AND c.RecordedDateTime <= i.index_date
WHERE c.RecordedDateTime IS NOT NULL
GROUP BY i.PersonId
"""
pancreatitis_history_during_lookback = spark.sql(pancreatitis_history_during_lookback_query)
pancreatitis_history_during_lookback.createOrReplaceTempView("tbl_pancreatitis_history")

In [ ]:
pancreatitis_outcome_time_status_query = """
WITH Censor_Calc AS (
    SELECT
        ind.PersonId,
        ind.index_date,
        event.RecordedDateTime,
        d.Death_Time,
        enc.Last_Encounter_Date,
        LEAST(
            DATE '2026-03-06',                                                      -- Administrative cutoff
            COALESCE(d.Death_Time,                          DATE '2026-03-06'),     -- Death
            CASE
                WHEN enc.Last_Encounter_Date >= ind.index_date
                THEN DATEADD(year, 1, enc.Last_Encounter_Date)
                ELSE DATE '2026-03-06'
            END,
            DATEADD(year, 3, ind.index_date)                                        -- 3yr max follow-up
        ) AS Censor_Date

    FROM tbl_patient_index_date ind
    LEFT JOIN tbl_first_pancreatitis_diagnosis event
        ON ind.PersonId = event.PersonId AND event.RecordedDateTime > ind.index_date
    LEFT JOIN tbl_death d
        ON ind.PersonId = d.PersonId
    LEFT JOIN tbl_last_encounter enc
        ON ind.PersonId = enc.PersonId
    WHERE ind.PersonId NOT IN (
        SELECT PersonId
        FROM tbl_pancreatitis_history
        WHERE pancreatitis_history > 0
    )
),
First_Step AS (
    SELECT
        *,

        -----------------------------------------------------------------------
        -- Event or Censor Date
        -----------------------------------------------------------------------
        CASE
            -- Invalid
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN NULL

            -- Event
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN RecordedDateTime

            -- Censored
            ELSE Censor_Date
        END AS Event_or_Censor_Date,

        -----------------------------------------------------------------------
        -- Outcome (1 = event, 0 = censored)
        -----------------------------------------------------------------------
        CASE
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 1
            ELSE 0
        END AS Outcome,

        -----------------------------------------------------------------------
        -- Outcome Text
        -----------------------------------------------------------------------
        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN 'Invalid'

            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 'Event'

            WHEN Death_Time IS NOT NULL
                 AND Censor_Date = Death_Time
            THEN 'Censored by Death'

            WHEN Last_Encounter_Date IS NOT NULL
                 AND Last_Encounter_Date >= index_date
                 AND Censor_Date = DATEADD(year, 1, Last_Encounter_Date)
            THEN 'Censored by Encounter Gap (1yr)'

            WHEN Censor_Date = DATEADD(year, 3, index_date)
            THEN 'Censored at Max Follow-up (3yr)'

            ELSE 'Censored at Administrative Cutoff'
        END AS Outcome_Text

    FROM Censor_Calc
)
SELECT
    *,
    DATEDIFF(day, index_date, Event_or_Censor_Date) AS Survival_Time
FROM First_Step
ORDER BY PersonId
"""
pancreatitis_outcome_time_status = spark.sql(pancreatitis_outcome_time_status_query)
pancreatitis_outcome_time_status.createOrReplaceTempView("tbl_pancreatitis_Endpoint")

In [ ]:
pancreatitis_outcome_time_status = pancreatitis_outcome_time_status.drop_duplicates()
save_path = "/aim_1b/Pancreatitis_Endpoint_t2d_secondary_female"
study.save_artifacts_data(df=pancreatitis_outcome_time_status, path=save_path, data_type="parquet", mode="overwrite")

## Urogenital infection

In [ ]:
ICD9_ui = ['599.0', '590', '595', '112.1', '112.2', '616.10']
ICD10_ui = ['N39.0', 'N30.0', 'N30.9', 'N10', 'B37.3', 'B37.31', 'B37.32', 'B37.4', 'B37.41', 'B37.42', 'B37.49']
ICD9_ui_codeset = snapshot.codeset("ICD9CM", 'selfAndDescendants', *ICD9_ui)
ICD10_ui_codeset = snapshot.codeset("ICD10CM", 'selfAndDescendants', *ICD10_ui)
ui_codeset = ps.concat([ICD9_ui_codeset, ICD10_ui_codeset])
ui_condition = snapshot.load_filtered_table('Condition', ui_codeset, apply_annulled_filter=True, view_name = "tbl_ui_diagnosis")

In [ ]:
first_ui_diagnosis_query = """
WITH Ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY PersonId
            ORDER BY RecordedDateTime ASC
        ) AS rn
    FROM tbl_ui_diagnosis
)
SELECT
    PersonId,
    RecordedDateTime
FROM Ranked
WHERE rn = 1;
"""
first_ui_event = spark.sql(first_ui_diagnosis_query)
first_ui_event.createOrReplaceTempView('tbl_first_ui_diagnosis')

In [ ]:
ui_history_during_lookback_query = """
SELECT
    i.PersonId,
    COUNT(c.RecordedDateTime) AS ui_history
FROM tbl_patient_index_date i
LEFT JOIN tbl_ui_diagnosis c ON i.PersonId = c.PersonId AND c.RecordedDateTime >= i.look_back AND c.RecordedDateTime <= i.index_date
WHERE c.RecordedDateTime IS NOT NULL
GROUP BY i.PersonId
"""
ui_history_during_lookback = spark.sql(ui_history_during_lookback_query)
ui_history_during_lookback.createOrReplaceTempView("tbl_ui_history")

In [ ]:
ui_outcome_time_status_query = """
WITH Censor_Calc AS (
    SELECT
        ind.PersonId,
        ind.index_date,
        event.RecordedDateTime,
        d.Death_Time,
        enc.Last_Encounter_Date,
        LEAST(
            DATE '2026-03-06',                                                      -- Administrative cutoff
            COALESCE(d.Death_Time,                          DATE '2026-03-06'),     -- Death
            CASE
                WHEN enc.Last_Encounter_Date >= ind.index_date
                THEN DATEADD(year, 1, enc.Last_Encounter_Date)
                ELSE DATE '2026-03-06'
            END,
            DATEADD(year, 3, ind.index_date)                                        -- 3yr max follow-up
        ) AS Censor_Date

    FROM tbl_patient_index_date ind
    LEFT JOIN tbl_first_ui_diagnosis event
        ON ind.PersonId = event.PersonId AND event.RecordedDateTime > ind.index_date
    LEFT JOIN tbl_death d
        ON ind.PersonId = d.PersonId
    LEFT JOIN tbl_last_encounter enc
        ON ind.PersonId = enc.PersonId
    WHERE ind.PersonId NOT IN (
        SELECT PersonId
        FROM tbl_ui_history
        WHERE ui_history > 0
    )
),
First_Step AS (
    SELECT
        *,

        -----------------------------------------------------------------------
        -- Event or Censor Date
        -----------------------------------------------------------------------
        CASE
            -- Invalid
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN NULL

            -- Event
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN RecordedDateTime

            -- Censored
            ELSE Censor_Date
        END AS Event_or_Censor_Date,

        -----------------------------------------------------------------------
        -- Outcome (1 = event, 0 = censored)
        -----------------------------------------------------------------------
        CASE
            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 1
            ELSE 0
        END AS Outcome,

        -----------------------------------------------------------------------
        -- Outcome Text
        -----------------------------------------------------------------------
        CASE
            WHEN (Death_Time IS NOT NULL AND index_date > Death_Time)
                 OR index_date > DATE '2026-03-06'
            THEN 'Invalid'

            WHEN RecordedDateTime IS NOT NULL
                 AND index_date < RecordedDateTime
                 AND RecordedDateTime <= Censor_Date
            THEN 'Event'

            WHEN Death_Time IS NOT NULL
                 AND Censor_Date = Death_Time
            THEN 'Censored by Death'

            WHEN Last_Encounter_Date IS NOT NULL
                 AND Last_Encounter_Date >= index_date
                 AND Censor_Date = DATEADD(year, 1, Last_Encounter_Date)
            THEN 'Censored by Encounter Gap (1yr)'

            WHEN Censor_Date = DATEADD(year, 3, index_date)
            THEN 'Censored at Max Follow-up (3yr)'

            ELSE 'Censored at Administrative Cutoff'
        END AS Outcome_Text

    FROM Censor_Calc
)
SELECT
    *,
    DATEDIFF(day, index_date, Event_or_Censor_Date) AS Survival_Time
FROM First_Step
ORDER BY PersonId
"""
ui_outcome_time_status = spark.sql(ui_outcome_time_status_query)
ui_outcome_time_status.createOrReplaceTempView("tbl_ui_Endpoint")

In [ ]:
ui_outcome_time_status = ui_outcome_time_status.drop_duplicates()
save_path = "/aim_1b/ui_Endpoint_t2d_secondary_female"
study.save_artifacts_data(df=ui_outcome_time_status, path=save_path, data_type="parquet", mode="overwrite")

# Dataset Export

In [ ]:
all_sex_base = study.load_artifacts_data("/output_data/T2D_Secondary_basedata_Mar10")
all_sex_base = all_sex_base.drop_duplicates()

race_map = {
    1067364: "White",
    1067319: "Black",
    1067294: "Asian",
    1066466: "Other Race",
    1067338: "Other Race",
    1067385: "Other Race"
}

ethnicity_map = {
    1065401: "Not Hispanic/Latino",
    1065359: "Hispanic/Latino"
}

all_sex_base['Race'] = all_sex_base['Race'].map(race_map).fillna("Unknown")
all_sex_base['Ethnicity'] = all_sex_base['Ethnicity'].map(ethnicity_map).fillna("Unknown")
all_sex_base['DcsiScoreGroup'] = all_sex_base['DcsiScore'].apply(lambda x: "0" if x == 0 else "+1")

In [ ]:
male_secondary_endpoint = study.load_artifacts_data("/aim_1b/CKD_Endpoint_t2d_secondary_male")
female_secondary_endpoint = study.load_artifacts_data("/aim_1b/CKD_Endpoint_t2d_secondary_female")

male_secondary_endpoint['Sex'] = 'M'
female_secondary_endpoint['Sex'] = 'F'
all_sex_endpoint = ps.concat([male_secondary_endpoint, female_secondary_endpoint]).drop_duplicates()

merged_df = all_sex_endpoint.merge(
    all_sex_base[['PersonId', 'FirstMedTime', 'FirstMedGroup', 'DcsiScoreGroup', 'Age', 'AgeGroup', 'Race', 'Ethnicity']],
    left_on=['PersonId', 'index_date'],
    right_on=['PersonId', 'FirstMedTime'],
    how='left'
)
merged_df.drop_duplicates()

save_path = "/aim_1b/CKD_Endpoint_t2d_secondary_all_sex_population"
study.save_artifacts_data(df=merged_df, path=save_path, data_type="parquet", mode="overwrite")